# M19d - Standard reservoir-task panel

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Submission evidence.** The four standard tasks are evaluated with the same current temperature-path protocol.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import tcr_core as tcr
REPRO=ROOT/'results'/'reproduced'; REPRO.mkdir(parents=True,exist_ok=True)
def bootstrap_mean(x,n_boot=30000,seed=1):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    rng=np.random.default_rng(seed); idx=rng.integers(0,len(x),size=(n_boot,len(x)))
    b=x[idx].mean(axis=1)
    return float(x.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))

In [2]:
SEED=20260718; TRIALS=8
TASKS=['lorenz_x','mackey_glass','memory_d10','narma10']
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=.8,ridge=1e-5)
rep=tcr.run_panel(TASKS,TRIALS,['temperature'],seed=SEED,**CONFIG); rep.to_csv(REPRO/'m19d_replication_case_metrics.csv',index=False)
ci=bootstrap_mean(rep.safe_gain,seed=1906)
summary=pd.DataFrame([{'n_cases':len(rep),'near_optimal_containment':rep.near_contained.mean(),'exact_oracle_containment':rep.exact_contained.mean(),'mean_safe_gain':ci[0],'safe_gain_ci_low':ci[1],'safe_gain_ci_high':ci[2],'mean_full_gain':rep.full_gain.mean(),'mean_safe_width':rep.safe_width.mean()}])
summary.to_csv(REPRO/'m19d_replication_summary.csv',index=False)
display(summary.round(6)); display(rep.groupby('task')[['safe_gain','full_gain','near_contained','exact_contained']].mean().round(6))

,n_cases,near_optimal_containment,exact_oracle_containment,mean_safe_gain,safe_gain_ci_low,safe_gain_ci_high,mean_full_gain,mean_safe_width
0,32,0.84375,0.6875,0.001585,0.000472,0.003062,0.007261,2.71875


,safe_gain,full_gain,near_contained,exact_contained
task,,,,
lorenz_x,0.000922,0.005786,0.875,0.75
mackey_glass,0.001978,0.005544,0.875,0.50
memory_d10,0.001191,0.005975,0.875,0.75
narma10,0.002247,0.011739,0.750,0.75
